# FD004 Paper 2 — Shift-V1 / DS-A with frozen P2-E8 specification

## Objective

Evaluate the selected **P2-E8 60-cycle point predictor** under the frozen Paper 1 **DS-A held-out operating-regime shift**.

### What remains frozen from Paper 1 DS-A
- held-out regime = 0
- engine-level train / calibration / test separation
- train/calibration prediction endpoints exclude the held-out regime
- test prediction endpoints contain the held-out regime
- sensor scaling is fit only on eligible training observations
- complete consecutive trajectory history is preserved

### What changes from the Paper 1 DS-A notebook
Only the selected Paper 2 point-model specification is substituted:
- temporal context: **60 cycles**
- input: 14 scaled sensors + 6 regime one-hot channels = 20
- projection: 20D → 18D, no bias
- core: LSTM(64)
- output: one RUL prediction
- optimizer/loss: Adam 1e-3 / MSE
- maximum epochs: 60
- batch size: 64
- early stopping patience: 10

This is a **point-model shift validation notebook only**. Do not run conformal calibration here. UQ-V1 is a separate later gate.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports and constants

In [2]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

DATA_DIR = "/content/drive/MyDrive/CMAPSSData"
OUTPUT_DIR = "/content/drive/MyDrive/FD004_Paper2_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RUL_CLIP = 125
WINDOW_SIZE = 60
N_REGIMES = 6

OPERATING_SETTINGS = [
    "setting_1",
    "setting_2",
    "setting_3"
]

SELECTED_SENSORS = [
    "s_2", "s_3", "s_4", "s_7", "s_8", "s_9",
    "s_11", "s_12", "s_13", "s_14",
    "s_15", "s_17", "s_20", "s_21"
]

ALL_RAW_COLS = (
    ["unit_id", "time_cycles"]
    + OPERATING_SETTINGS
    + [f"s_{i}" for i in range(1, 22)]
)

print("DS-A configuration loaded.")
print(f"Seed: {SEED}")
print(f"Number of regimes: {N_REGIMES}")
print(f"Window size: {WINDOW_SIZE}")


DS-A configuration loaded.
Seed: 42
Number of regimes: 6
Window size: 60


##Load raw FD004 training data


In [3]:

train_path = os.path.join(DATA_DIR, "train_FD004.txt")

train_raw = pd.read_csv(
    train_path,
    sep=r"\s+",
    header=None
)

# C-MAPSS files may contain trailing empty columns
train_raw = train_raw.iloc[:, :len(ALL_RAW_COLS)]
train_raw.columns = ALL_RAW_COLS

print("FD004 training data loaded.")
print("Shape:", train_raw.shape)
print("Number of engines:", train_raw["unit_id"].nunique())

display(train_raw.head())


FD004 training data loaded.
Shape: (61249, 26)
Number of engines: 249


,unit_id,time_cycles,setting_1,setting_2,setting_3,s_1,s_2,s_3,s_4,s_5,...,s_12,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20,s_21
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,129.78,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,312.59,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,129.62,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,129.80,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,164.11,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754


## Calculate training RUL

In [4]:
max_cycles = (
    train_raw.groupby("unit_id")["time_cycles"]
    .max()
    .rename("max_cycle")
)

train_raw = train_raw.merge(
    max_cycles,
    on="unit_id",
    how="left"
)

train_raw["RUL_raw"] = (
    train_raw["max_cycle"] - train_raw["time_cycles"]
)

train_raw["RUL"] = train_raw["RUL_raw"].clip(
    upper=RUL_CLIP
)

print(train_raw[
    ["unit_id", "time_cycles", "RUL_raw", "RUL"]
].head(10))


   unit_id  time_cycles  RUL_raw  RUL
0        1            1      320  125
1        1            2      319  125
2        1            3      318  125
3        1            4      317  125
4        1            5      316  125
5        1            6      315  125
6        1            7      314  125
7        1            8      313  125
8        1            9      312  125
9        1           10      311  125


## Fit the six operating regimes


In [5]:
setting_scaler = StandardScaler()

settings_scaled = setting_scaler.fit_transform(
    train_raw[OPERATING_SETTINGS]
)

kmeans = KMeans(
    n_clusters=N_REGIMES,
    random_state=SEED,
    n_init=10
)

train_raw["regime"] = kmeans.fit_predict(
    settings_scaled
)

print("Operating regimes assigned.")
print(
    train_raw["regime"]
    .value_counts()
    .sort_index()
)


Operating regimes assigned.
regime
0    15395
1     9224
2     9139
3     9091
4     9238
5     9162
Name: count, dtype: int64


## Create the regime summary

In [6]:
regime_summary = (
    train_raw
    .groupby("regime")
    .agg(
        observations=("unit_id", "size"),
        engines=("unit_id", "nunique"),
        mean_RUL=("RUL", "mean"),
        median_RUL=("RUL", "median"),
        mean_setting_1=("setting_1", "mean"),
        mean_setting_2=("setting_2", "mean"),
        mean_setting_3=("setting_3", "mean")
    )
    .reset_index()
)

regime_summary["observation_pct"] = (
    100
    * regime_summary["observations"]
    / len(train_raw)
)

display(
    regime_summary.round(3)
)


,regime,observations,engines,mean_RUL,median_RUL,mean_setting_1,mean_setting_2,mean_setting_3,observation_pct
0,0,15395,249,92.759,123.0,42.003,0.840,100.0,25.135
1,1,9224,249,93.098,123.0,10.003,0.251,100.0,15.060
2,2,9139,249,93.273,122.0,25.003,0.621,60.0,14.921
3,3,9091,249,93.099,123.0,20.003,0.701,100.0,14.843
4,4,9238,249,92.744,121.0,0.002,0.000,100.0,15.083
5,5,9162,249,93.095,122.0,35.003,0.841,100.0,14.959


## Determine engine/regime overlap


In [7]:
engine_regime_counts = (
    train_raw
    .groupby("unit_id")["regime"]
    .nunique()
)

print("Regimes encountered per engine:")
display(
    engine_regime_counts
    .value_counts()
    .sort_index()
    .rename_axis("number_of_regimes")
    .reset_index(name="number_of_engines")
)

print(
    "\nMedian regimes per engine:",
    engine_regime_counts.median()
)

print(
    "Maximum regimes per engine:",
    engine_regime_counts.max()
)


Regimes encountered per engine:


,number_of_regimes,number_of_engines
0,6,249



Median regimes per engine: 6.0
Maximum regimes per engine: 6


## Set held-out regime


In [8]:
HELD_OUT_REGIME = 0

print(f"DS-A held-out operating regime: {HELD_OUT_REGIME}")

heldout_rows = train_raw[
    train_raw["regime"] == HELD_OUT_REGIME
]

print("Held-out observations:", len(heldout_rows))
print(
    "Percentage of dataset:",
    round(100 * len(heldout_rows) / len(train_raw), 2),
    "%"
)
print(
    "Engines represented:",
    heldout_rows["unit_id"].nunique()
)


DS-A held-out operating regime: 0
Held-out observations: 15395
Percentage of dataset: 25.14 %
Engines represented: 249


## Create engine-level split

In [9]:
from sklearn.model_selection import train_test_split

all_engines = np.array(
    sorted(train_raw["unit_id"].unique())
)

train_engines, temp_engines = train_test_split(
    all_engines,
    test_size=0.30,
    random_state=SEED
)

cal_engines, test_engines = train_test_split(
    temp_engines,
    test_size=0.50,
    random_state=SEED
)

print("Train engines:", len(train_engines))
print("Calibration engines:", len(cal_engines))
print("Test engines:", len(test_engines))

print(
    "\nTotal:",
    len(train_engines)
    + len(cal_engines)
    + len(test_engines)
)


Train engines: 174
Calibration engines: 37
Test engines: 38

Total: 249


## Verify zero engine leakage

In [10]:
assert len(
    set(train_engines) & set(cal_engines)
) == 0

assert len(
    set(train_engines) & set(test_engines)
) == 0

assert len(
    set(cal_engines) & set(test_engines)
) == 0

print("✓ No engine leakage between splits.")


✓ No engine leakage between splits.


## Apply the DS-A regime exclusion

In [11]:
train_ds = train_raw[
    train_raw["unit_id"].isin(train_engines)
].copy()

cal_ds = train_raw[
    train_raw["unit_id"].isin(cal_engines)
].copy()

test_ds = train_raw[
    train_raw["unit_id"].isin(test_engines)
].copy()


# Remove held-out regime from TRAIN
train_ds = train_ds[
    train_ds["regime"] != HELD_OUT_REGIME
].copy()

# Remove held-out regime from CALIBRATION
cal_ds = cal_ds[
    cal_ds["regime"] != HELD_OUT_REGIME
].copy()

# TEST contains only held-out regime observations
test_ds = test_ds[
    test_ds["regime"] == HELD_OUT_REGIME
].copy()


In [12]:
print("TRAIN")
print(train_ds["regime"].value_counts().sort_index())

print("\nCALIBRATION")
print(cal_ds["regime"].value_counts().sort_index())

print("\nTEST")
print(test_ds["regime"].value_counts().sort_index())


TRAIN
regime
1    6409
2    6378
3    6354
4    6491
5    6438
Name: count, dtype: int64

CALIBRATION
regime
1    1385
2    1401
3    1339
4    1347
5    1351
Name: count, dtype: int64

TEST
regime
0    2352
Name: count, dtype: int64


## Hard validation

In [13]:
assert HELD_OUT_REGIME not in train_ds["regime"].unique()
assert HELD_OUT_REGIME not in cal_ds["regime"].unique()

assert set(
    test_ds["regime"].unique()
) == {HELD_OUT_REGIME}

assert set(
    train_ds["unit_id"].unique()
).isdisjoint(
    set(test_ds["unit_id"].unique())
)

assert set(
    cal_ds["unit_id"].unique()
).isdisjoint(
    set(test_ds["unit_id"].unique())
)

print("✓ DS-A split validation passed.")
print("✓ Held-out regime absent from training.")
print("✓ Held-out regime absent from calibration.")
print("✓ Test contains held-out regime only.")
print("✓ Engine-level isolation preserved.")


✓ DS-A split validation passed.
✓ Held-out regime absent from training.
✓ Held-out regime absent from calibration.
✓ Test contains held-out regime only.
✓ Engine-level isolation preserved.


## Fit sensor scaling on eligible training observations only

In [14]:
# ============================================================
# DS-A preprocessing
# Fit sensor scaler using TRAIN ENGINES and NON-HELD-OUT
# endpoint-eligible operating conditions only
# ============================================================

sensor_scaler = StandardScaler()

scaler_fit_df = train_raw[
    (train_raw["unit_id"].isin(train_engines)) &
    (train_raw["regime"] != HELD_OUT_REGIME)
].copy()

sensor_scaler.fit(
    scaler_fit_df[SELECTED_SENSORS]
)

print("Sensor scaler fitted.")
print("Rows used:", len(scaler_fit_df))
print("Engines used:", scaler_fit_df["unit_id"].nunique())


Sensor scaler fitted.
Rows used: 32070
Engines used: 174


## Scale the original trajectories

In [15]:
# Keep complete trajectories intact.
# Only the sensor values are transformed.

trajectory_df = train_raw.copy()

trajectory_df[SELECTED_SENSORS] = sensor_scaler.transform(
    trajectory_df[SELECTED_SENSORS]
)

trajectory_df = trajectory_df.sort_values(
    ["unit_id", "time_cycles"]
).reset_index(drop=True)

print("Full trajectories prepared:", trajectory_df.shape)


Full trajectories prepared: (61249, 30)


## Leakage-safe window function

In [16]:
def create_ds_a_windows(
    df,
    engine_ids,
    endpoint_mode,
    held_out_regime=0,
    window_size=60
):
    """
    Build windows from uninterrupted original engine trajectories.

    endpoint_mode:
        'seen'    -> endpoint regime must NOT be held out
        'heldout' -> endpoint regime MUST be held out

    Returns
    -------
    X_sensors : (N, window_size, 14)
    X_regimes : (N, window_size)
    y         : (N,)
    ids       : (N,)
    cycles    : endpoint cycle for each window
    endpoint_regimes : regime at prediction endpoint
    """

    X_sensors = []
    X_regimes = []
    y = []
    ids = []
    cycles = []
    endpoint_regimes = []

    for engine_id in sorted(engine_ids):

        eng = (
            df[df["unit_id"] == engine_id]
            .sort_values("time_cycles")
            .reset_index(drop=True)
        )

        if len(eng) < window_size:
            continue

        for end_idx in range(window_size - 1, len(eng)):

            start_idx = end_idx - window_size + 1

            window = eng.iloc[start_idx:end_idx + 1]

            endpoint_regime = int(
                eng.iloc[end_idx]["regime"]
            )

            # Eligibility depends ONLY on prediction endpoint
            if endpoint_mode == "seen":
                if endpoint_regime == held_out_regime:
                    continue

            elif endpoint_mode == "heldout":
                if endpoint_regime != held_out_regime:
                    continue

            else:
                raise ValueError(
                    "endpoint_mode must be 'seen' or 'heldout'"
                )

            # Ensure cycles are genuinely consecutive
            cycle_values = window["time_cycles"].to_numpy()

            if not np.all(np.diff(cycle_values) == 1):
                raise ValueError(
                    f"Non-consecutive window detected "
                    f"for engine {engine_id}"
                )

            X_sensors.append(
                window[SELECTED_SENSORS].to_numpy(
                    dtype=np.float32
                )
            )

            X_regimes.append(
                window["regime"].to_numpy(
                    dtype=np.int32
                )
            )

            y.append(
                float(eng.iloc[end_idx]["RUL"])
            )

            ids.append(engine_id)

            cycles.append(
                int(eng.iloc[end_idx]["time_cycles"])
            )

            endpoint_regimes.append(
                endpoint_regime
            )

    return (
        np.asarray(X_sensors, dtype=np.float32),
        np.asarray(X_regimes, dtype=np.int32),
        np.asarray(y, dtype=np.float32),
        np.asarray(ids, dtype=np.int32),
        np.asarray(cycles, dtype=np.int32),
        np.asarray(endpoint_regimes, dtype=np.int32)
    )


## Construct train, calibration and DS-A test windows

In [17]:
(
    X_train_sensors,
    reg_train,
    y_train_ds,
    ids_train,
    cycles_train,
    reg_train_endpoint
) = create_ds_a_windows(
    trajectory_df,
    train_engines,
    endpoint_mode="seen",
    held_out_regime=HELD_OUT_REGIME,
    window_size=WINDOW_SIZE
)


(
    X_cal_sensors,
    reg_cal,
    y_cal_ds,
    ids_cal,
    cycles_cal,
    reg_cal_endpoint
) = create_ds_a_windows(
    trajectory_df,
    cal_engines,
    endpoint_mode="seen",
    held_out_regime=HELD_OUT_REGIME,
    window_size=WINDOW_SIZE
)


(
    X_test_sensors,
    reg_test,
    y_test_ds,
    ids_test,
    cycles_test,
    reg_test_endpoint
) = create_ds_a_windows(
    trajectory_df,
    test_engines,
    endpoint_mode="heldout",
    held_out_regime=HELD_OUT_REGIME,
    window_size=WINDOW_SIZE
)


print("TRAIN:", X_train_sensors.shape, y_train_ds.shape)
print("CAL  :", X_cal_sensors.shape, y_cal_ds.shape)
print("TEST :", X_test_sensors.shape, y_test_ds.shape)


TRAIN: (24429, 60, 14) (24429,)
CAL  : (5167, 60, 14) (5167,)
TEST : (1753, 60, 14) (1753,)


## Validate the windows

In [18]:
print("Train endpoint regimes:",
      np.unique(reg_train_endpoint))

print("Calibration endpoint regimes:",
      np.unique(reg_cal_endpoint))

print("Test endpoint regimes:",
      np.unique(reg_test_endpoint))


assert HELD_OUT_REGIME not in np.unique(
    reg_train_endpoint
)

assert HELD_OUT_REGIME not in np.unique(
    reg_cal_endpoint
)

assert set(
    np.unique(reg_test_endpoint)
) == {HELD_OUT_REGIME}


assert set(ids_train).isdisjoint(
    set(ids_cal)
)

assert set(ids_train).isdisjoint(
    set(ids_test)
)

assert set(ids_cal).isdisjoint(
    set(ids_test)
)

assert X_train_sensors.shape[1] == WINDOW_SIZE
assert X_cal_sensors.shape[1] == WINDOW_SIZE
assert X_test_sensors.shape[1] == WINDOW_SIZE

print("\n✓ Window construction passed.")
print("✓ No engine leakage.")
print("✓ Train/cal endpoints exclude held-out regime.")
print("✓ Test endpoints contain held-out regime only.")
print("✓ Every sequence contains 60 consecutive cycles.")


Train endpoint regimes: [1 2 3 4 5]
Calibration endpoint regimes: [1 2 3 4 5]
Test endpoint regimes: [0]

✓ Window construction passed.
✓ No engine leakage.
✓ Train/cal endpoints exclude held-out regime.
✓ Test endpoints contain held-out regime only.
✓ Every sequence contains 60 consecutive cycles.


In [19]:
import tensorflow as tf

reg_train_oh = tf.keras.utils.to_categorical(
    reg_train,
    num_classes=N_REGIMES
)

reg_cal_oh = tf.keras.utils.to_categorical(
    reg_cal,
    num_classes=N_REGIMES
)

reg_test_oh = tf.keras.utils.to_categorical(
    reg_test,
    num_classes=N_REGIMES
)

X_train_ds = np.concatenate(
    [X_train_sensors, reg_train_oh],
    axis=-1
)

X_cal_ds = np.concatenate(
    [X_cal_sensors, reg_cal_oh],
    axis=-1
)

X_test_ds = np.concatenate(
    [X_test_sensors, reg_test_oh],
    axis=-1
)

print("Final model inputs:")
print("Train:", X_train_ds.shape)
print("Cal  :", X_cal_ds.shape)
print("Test :", X_test_ds.shape)

assert X_train_ds.shape[-1] == 20

print("✓ 14 sensor channels + 6 regime channels = 20 features.")


Final model inputs:
Train: (24429, 60, 20)
Cal  : (5167, 60, 20)
Test : (1753, 60, 20)
✓ 14 sensor channels + 6 regime channels = 20 features.


## P2-E8 point-model training protocol for Shift-V1

The **architecture and hyperparameters are frozen**, but the weights are retrained using only the DS-A-allowed training distribution, matching the logic of the original Paper 1 shift experiment.

The calibration-engine split is used here only as a leakage-safe **validation set for early stopping**. Because it is touched by model selection, do **not** reuse these same predictions as conformal calibration data later. UQ-V1 must use its own untouched calibration protocol.


In [20]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense, LSTM
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

EPOCHS = 60
BATCH_SIZE = 64
PATIENCE = 10

def set_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def build_p2e8_point_model():
    inp = Input(shape=(WINDOW_SIZE, 20), name="p2e8_input")
    x = Dense(18, use_bias=False, name="linear_projection_18d")(inp)
    x = LSTM(64, name="lstm_core")(x)
    out = Dense(1, name="rul_head")(x)

    model = Model(inp, out, name="P2_E8_DS_A_60cycle")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae"]
    )
    return model

set_all_seeds(SEED)
p2e8_model = build_p2e8_point_model()
p2e8_model.summary()


Model: "P2_E8_DS_A_60cycle"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ p2e8_input (InputLayer)         │ (None, 60, 20)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ linear_projection_18d (Dense)   │ (None, 60, 18)         │           360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_core (LSTM)                │ (None, 64)             │        21,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rul_head (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,673 (84.66 KB)

 Trainable params: 21,673 (84.66 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
set_all_seeds(SEED)

DS_A_CKPT = os.path.join(OUTPUT_DIR, "P2_E8_DS_A_best.keras")

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        DS_A_CKPT,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=0
    )
]

history = p2e8_model.fit(
    X_train_ds,
    y_train_ds,
    validation_data=(X_cal_ds, y_cal_ds),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
    shuffle=True
)

print("Training stopped after:", len(history.history["loss"]), "epochs")
print("Best checkpoint:", DS_A_CKPT)


Epoch 1/60
382/382 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - loss: 5752.7104 - mae: 65.4114 - val_loss: 4290.5190 - val_mae: 56.2748
Epoch 2/60
382/382 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 3391.7632 - mae: 50.3666 - val_loss: 2754.7998 - val_mae: 46.1193
Epoch 3/60
382/382 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 2341.1379 - mae: 42.9622 - val_loss: 2081.1479 - val_mae: 41.0470
Epoch 4/60
382/382 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 1921.2394 - mae: 39.4932 - val_loss: 1843.2328 - val_mae: 38.7959
Epoch 5/60
382/382 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 1790.9926 - mae: 38.0498 - val_loss: 1780.1599 - val_mae: 37.8981
Epoch 6/60
382/382 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 1761.8596 - mae: 37.5102 - val_loss: 1768.0073 - val_mae: 37.5794
Epoch 7/60
382/382 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 1757.4314 - mae: 37.3288 - val_loss: 1766.1794 - val_mae: 37.4799
Epoch 8/60
382/382 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 1524.7369 - mae: 34.5163 - val_loss: 847.1945 - val_mae

## Shift-V1 / DS-A point-model evaluation and export

The held-out DS-A test set is used **once for final evaluation**. The export below is the file expected by the main Paper 2 notebook's Shift-V3 section.


In [22]:
pred_test = p2e8_model.predict(
    X_test_ds,
    batch_size=512,
    verbose=0
).reshape(-1)

test_rmse = float(np.sqrt(mean_squared_error(y_test_ds, pred_test)))
test_mae = float(mean_absolute_error(y_test_ds, pred_test))
test_bias = float(np.mean(pred_test - y_test_ds))

print("=" * 60)
print("P2-E8 SHIFT-V1 / DS-A POINT PERFORMANCE")
print("=" * 60)
print(f"RMSE : {test_rmse:.3f}")
print(f"MAE  : {test_mae:.3f}")
print(f"Bias : {test_bias:.3f}")
print(f"N    : {len(y_test_ds)}")
print(f"Paper 1 DS-A point-RMSE reference: 24.869")
print(f"RMSE change vs Paper 1 reference: {test_rmse - 24.869:+.3f}")

life_stage = np.where(
    y_test_ds > 80, "Early",
    np.where(y_test_ds > 40, "Mid", "Late")
)

stage_rows = []
for stage in ["Early", "Mid", "Late"]:
    mask = life_stage == stage
    if not np.any(mask):
        continue
    stage_rows.append({
        "Life_Stage": stage,
        "N": int(mask.sum()),
        "RMSE": float(np.sqrt(mean_squared_error(y_test_ds[mask], pred_test[mask]))),
        "MAE": float(mean_absolute_error(y_test_ds[mask], pred_test[mask])),
        "Bias_pred_minus_true": float(np.mean(pred_test[mask] - y_test_ds[mask]))
    })

ds_a_stage_metrics = pd.DataFrame(stage_rows)
display(ds_a_stage_metrics)


P2-E8 SHIFT-V1 / DS-A POINT PERFORMANCE
RMSE : 16.942
MAE  : 12.316
Bias : -1.450
N    : 1753
Paper 1 DS-A point-RMSE reference: 24.869
RMSE change vs Paper 1 reference: -7.927


,Life_Stage,N,RMSE,MAE,Bias_pred_minus_true
0,Early,967,17.040207,12.468933,-7.994668
1,Mid,385,22.182599,17.395927,8.132404
2,Late,401,9.065539,7.068632,5.134314


In [23]:
ds_a_export = pd.DataFrame({
    "unit_id": np.asarray(ids_test).astype(int),
    "cycle": np.asarray(cycles_test).astype(int),
    "RUL": np.asarray(y_test_ds, dtype=float),
    "prediction": np.asarray(pred_test, dtype=float),
    "regime": np.asarray(reg_test_endpoint).astype(int)
})

DS_A_EXPORT = os.path.join(
    OUTPUT_DIR,
    "DS_A_P2E8_predictions.csv"
)

DS_A_STAGE_EXPORT = os.path.join(
    OUTPUT_DIR,
    "DS_A_P2E8_stage_metrics.csv"
)

ds_a_export.to_csv(DS_A_EXPORT, index=False)
ds_a_stage_metrics.to_csv(DS_A_STAGE_EXPORT, index=False)

print("Saved prediction export:", DS_A_EXPORT)
print("Saved stage metrics:", DS_A_STAGE_EXPORT)
print("Rows:", len(ds_a_export))
display(ds_a_export.head())

required = {"unit_id", "cycle", "RUL", "prediction", "regime"}
assert required.issubset(ds_a_export.columns)
assert len(ds_a_export) == len(y_test_ds)
print("PASS: DS-A export is ready for the main Paper 2 Shift-V3 notebook.")


Saved prediction export: /content/drive/MyDrive/FD004_Paper2_Results/DS_A_P2E8_predictions.csv
Saved stage metrics: /content/drive/MyDrive/FD004_Paper2_Results/DS_A_P2E8_stage_metrics.csv
Rows: 1753


,unit_id,cycle,RUL,prediction,regime
0,17,64,87.0,122.615669,0
1,17,69,82.0,120.370583,0
2,17,82,69.0,115.488243,0
3,17,83,68.0,111.865723,0
4,17,87,64.0,103.897652,0


PASS: DS-A export is ready for the main Paper 2 Shift-V3 notebook.
